In [1]:
import sys
import os

os.chdir("..")
sys.path.append(".")

print("Working directory:", os.getcwd())

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend


In [2]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from config import settings
import json

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

print("Imports OK")

Imports OK


In [5]:
creds = None

if os.path.exists(settings.google_token_file):
    creds = Credentials.from_authorized_user_file(settings.google_token_file, SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            settings.google_credentials_file, SCOPES
        )
        creds = flow.run_local_server(port=0)

    with open(settings.google_token_file, "w") as f:
        f.write(creds.to_json())

print("Authentication OK")
print("Token saved to:", settings.google_token_file)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=442337534295-qq9qjjfook0n12jbe2tpnbuc0o9c90bq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50444%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=SkxOhY5b4ZkPVaD98ZCVoOEj6zlfgW&code_challenge=gKkjSxCBUV3UhSejT2wOANMO_qtj7lHMLa_-rUgX2wI&code_challenge_method=S256&access_type=offline
Authentication OK
Token saved to: token.json


In [4]:
service = build("drive", "v3", credentials=creds)

# List folders inside the root 'contracts' folder
# First, find the root folder by name
response = service.files().list(
    q=f"name='{settings.google_drive_root_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id, name)"
).execute()

folders = response.get("files", [])
print("Root folders found:", folders)

Root folders found: [{'id': '1oUcCX5c2ILwv_gMMTLc1nBEpP8v6Ls_5', 'name': 'contracts'}]


In [7]:
root_folder_id = folders[0]["id"]

subfolders = service.files().list(
    q=f"'{root_folder_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id, name)"
).execute().get("files", [])

print("Project folders:")
for f in subfolders:
    print(f"  - {f['name']} ({f['id']})")

Project folders:
  - QMC Heritage (1wpUnTaBlpwK7fppBOGz8kvRVDHkBN4VY)


In [8]:
print("Files per project:")
for folder in subfolders:
    files = service.files().list(
        q=f"'{folder['id']}' in parents and trashed=false",
        fields="files(id, name, mimeType, modifiedTime)"
    ).execute().get("files", [])
    
    print(f"\n{folder['name']}:")
    for f in files:
        print(f"  - {f['name']} ({f['mimeType']})")

Files per project:

QMC Heritage:
  - Contract C2024-49.pdf (application/pdf)


In [12]:
import sys
import os

# Always anchor to backend/ regardless of where Jupyter started
os.chdir(os.path.dirname(os.path.abspath("config.py")))

# If we're still not in backend/, find it explicitly
if not os.path.exists("config.py"):
    os.chdir("backend")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))
print("token.json exists:", os.path.exists("token.json"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True
token.json exists: True


In [13]:
import io
import pdfplumber
from docx import Document
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from config import settings

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_authorized_user_file(settings.google_token_file, SCOPES)
service = build("drive", "v3", credentials=creds)

print("Imports OK")

Imports OK


In [17]:
from googleapiclient.http import MediaIoBaseDownload

def download_file(file_id: str) -> bytes:
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return buffer.getvalue()

# Use the file ID from your Drive listing in the previous notebook
PDF_FILE_ID = "1c-18lHMiW-wxFnUlhd6y21RNlP6ex_Ek"

print("Download function ready")

Download function ready


In [18]:
# Download the PDF into memory
pdf_bytes = download_file(PDF_FILE_ID)
print(f"Downloaded {len(pdf_bytes):,} bytes")

# Extract text using pdfplumber
with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    pages = []
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            pages.append(text)
            print(f"Page {i+1}: {len(text)} chars")
        else:
            print(f"Page {i+1}: empty")

full_text = "\n".join(pages)
print(f"\nTotal extracted: {len(full_text):,} chars")
print("\n--- First 500 chars ---")
print(full_text[:500])

Downloaded 1,341,352 bytes
Page 1: 229 chars
Page 2: 3933 chars
Page 3: 4401 chars
Page 4: 3921 chars
Page 5: 4312 chars
Page 6: 4041 chars
Page 7: 4327 chars
Page 8: 446 chars
Page 9: 2841 chars
Page 10: 3324 chars
Page 11: 2970 chars
Page 12: 3782 chars
Page 13: 3736 chars
Page 14: 3871 chars
Page 15: 3615 chars
Page 16: 3442 chars
Page 17: 3647 chars
Page 18: 3598 chars
Page 19: 4022 chars
Page 20: 3883 chars
Page 21: 3556 chars
Page 22: 3317 chars
Page 23: 3357 chars
Page 24: 3718 chars
Page 25: 3243 chars
Page 26: 3682 chars
Page 27: 3440 chars
Page 28: 3647 chars
Page 29: 3431 chars
Page 30: 3360 chars
Page 31: 3877 chars
Page 32: 3471 chars
Page 33: 3698 chars
Page 34: 3113 chars
Page 35: 2947 chars
Page 36: 3962 chars
Page 37: 3887 chars
Page 38: 3209 chars
Page 39: 3435 chars
Page 40: 3456 chars
Page 41: 3329 chars
Page 42: 3922 chars
Page 43: 3461 chars
Page 44: 3095 chars
Page 45: 3897 chars
Page 46: 3953 chars
Page 47: 3410 chars
Page 48: 2960 chars
Page 49: 3078 chars
Page

In [16]:
for folder in subfolders:
    files = service.files().list(
        q=f"'{folder['id']}' in parents and trashed=false",
        fields="files(id, name, mimeType)"
    ).execute().get("files", [])
    
    for f in files:
        print(f"Name: {f['name']}")
        print(f"ID:   {f['id']}")
        print()

Name: Contract C2024-49.pdf
ID:   1c-18lHMiW-wxFnUlhd6y21RNlP6ex_Ek



In [19]:
print(f"Total pages: {len(pdf.pages)}")
print(f"Total chars: {len(full_text):,}")
print("\n--- First 1000 chars ---")
print(full_text[:1000])

Total pages: 117
Total chars: 380,054

--- First 1000 chars ---
ةماعلا لاغشلأا ةئيه
PUBLIC WORKS AUTHORITY
رطق ةلود
STATE OF QATAR
ةسدنهلاو ءانبلا لامعأ
ءانبلاو ميمصتلا دقعل ةماعلا طورشلا
2018 ةعبط
BUILDING AND ENGINEERING WORKS
GENERAL CONDITIONS OF CONTRACT FOR
DESIGN AND BUILD
2018 EDITION
STATE OF QATAR DESIGN AND BUILD
PUBLIC WORKS AUTHORITY GENERAL CONDITIONS OF CONTRACT
CONTENTS
1. THE CONTRACT AND PRELIMINARY MATTERS 1
1.1 Definitions and Interpretation .............................................................................................. 1
1.2 Contract Documents ........................................................................................................... 1
1.3 Preliminary Matters ............................................................................................................. 1
1.4 President as Authority’s Representative .............................................................................. 2
1.5 Contractor’s Representative ...........

In [1]:
DELETE FROM contract_chunks;
DELETE FROM ingested_files;

SyntaxError: invalid syntax (3757287453.py, line 1)

In [3]:
result = sb.table("contract_chunks") \
    .select("clause_ref, content") \
    .eq("filename", "Scope of Works.pdf") \
    .limit(10) \
    .execute()

for row in result.data:
    print(f"Clause: {row['clause_ref']}")
    print(row['content'][:200])
    print()

NameError: name 'sb' is not defined